In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Cài đặt thư viện

In [ ]:
!pip install -q openai-whisper torch torchaudio librosa soundfile transformers sentencepiece pydub noisereduce accelerate sentencepiece bitsandbytes silero-vad

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 116.5 MB/s eta 0:00:00


## 2. Import các thư viện

In [ ]:
import torch
import whisper
import librosa
import noisereduce as nr
import soundfile as sf
import numpy as np
from pydub import AudioSegment
from pydub.silence import split_on_silence, detect_silence
import io
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import time
from google.colab import files
import os
from silero_vad import load_silero_vad, get_speech_timestamps
import re

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## 3. Import mô hình Finetune

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_finetuned_small = whisper.load_model('small')
state_dict = torch.load('/content/drive/MyDrive/whisper_finetuned.pt', map_location='cpu')
model_finetuned_small.load_state_dict(state_dict)
model_finetuned_small.to(device)
model_finetuned_small.eval()

Whisper(
  (encoder): AudioEncoder(
    (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
    (blocks): ModuleList(
      (0-11): 12 x ResidualAttentionBlock(
        (attn): MultiHeadAttention(
          (query): Linear(in_features=768, out_features=768, bias=True)
          (key): Linear(in_features=768, out_features=768, bias=False)
          (value): Linear(in_features=768, out_features=768, bias=True)
          (out): Linear(in_features=768, out_features=768, bias=True)
        )
        (attn_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
        (mlp_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-0

In [ ]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def load_silero_vad():
    model, utils = torch.hub.load(
        repo_or_dir="snakers4/silero-vad",
        model="silero_vad",
        trust_repo=True
    )

    get_speech_timestamps = utils[0]

    return model, get_speech_timestamps

vad_model, get_speech_timestamps = load_silero_vad()

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip


## 4. Giảm nhiễu

In [ ]:
def reduce_audio_noise(audio_input, sr=None):
    if isinstance(audio_input, str):
        audio_data, sampling_rate = librosa.load(audio_input, sr=sr)
    else:
        audio_data = audio_input
        sampling_rate = sr

    reduced_noise_audio = nr.reduce_noise(y=audio_data, sr=sampling_rate)

    return reduced_noise_audio, sampling_rate

## 5. Chia đoạn âm thanh

In [ ]:
def split_audio_into_chunks(reduced_audio, sr, target_chunk_seconds):
    audio_int16 = (reduced_audio * 32767).astype(np.int16)
    audio_segment = AudioSegment(
        audio_int16.tobytes(),
        frame_rate=sr,
        sample_width=2,
        channels=1
    )

    silences = detect_silence(
        audio_segment,
        min_silence_len=300,
        silence_thresh=-40,
        seek_step=10
    )

    chunks = []
    chunk_start = 0
    target_ms = target_chunk_seconds * 1000
    min_chunk_ms = 5000
    max_chunk_ms = 30000

    for silence_start, silence_end in silences:
        silence_mid = (silence_start + silence_end) // 2
        chunk_duration = silence_mid - chunk_start

        if chunk_duration >= target_ms:
            if chunk_duration > max_chunk_ms:
                while chunk_start < silence_mid:
                    end = min(chunk_start + max_chunk_ms, silence_mid)
                    chunks.append(audio_segment[chunk_start:end])
                    chunk_start = end
            else:
                chunks.append(audio_segment[chunk_start:silence_mid])
                chunk_start = silence_mid

    if chunk_start < len(audio_segment):
        last_chunk = audio_segment[chunk_start:]
        if len(last_chunk) >= min_chunk_ms:
            chunks.append(last_chunk)
        elif chunks:
            chunks[-1] = chunks[-1] + last_chunk
        else:
            chunks.append(last_chunk)

    return chunks

## 6. Xử lý đoạn âm thanh

In [ ]:
def remove_silence(audio_array, vad_model, get_speech_timestamps, sample_rate=16000):

    audio_tensor = torch.from_numpy(audio_array)

    speech_timestamps = get_speech_timestamps(
        audio_tensor,
        vad_model,
        sampling_rate=sample_rate
    )

    if len(speech_timestamps) == 0:
        return np.array([])

    speech_audio = []

    for ts in speech_timestamps:
        start = ts["start"]
        end = ts["end"]

        speech_audio.append(audio_array[start:end])

    return np.concatenate(speech_audio)

In [ ]:
def process_audio_chunks(chunks, whisper_model):
    processed_texts = []
    total_chunks = len(chunks)

    previous_context = ""
    max_context_words = 50
    current_audio_time_ms = 0

    total_speech_duration_sec = 0.0

    pending_samples = np.array([], dtype=np.float32)

    for i, chunk in enumerate(chunks):
        start_process_time = time.time()

        chunk_duration_ms = len(chunk)
        start_sec = current_audio_time_ms / 1000.0
        end_sec = (current_audio_time_ms + chunk_duration_ms) / 1000.0

        chunk_16k = chunk.set_frame_rate(16000).set_channels(1)
        samples = np.array(chunk_16k.get_array_of_samples()).astype(np.float32)
        max_val = float(1 << (8 * chunk_16k.sample_width - 1))
        samples = samples / max_val

        cleaned_samples = remove_silence(
            samples,
            vad_model,
            get_speech_timestamps
        )

        if cleaned_samples.size == 0:
            print(f"Chunk {i+1}: bỏ qua vì không có speech")
            current_audio_time_ms += chunk_duration_ms
            continue

        current_chunk_speech_sec = len(cleaned_samples) / 16000.0
        total_speech_duration_sec += current_chunk_speech_sec

        if cleaned_samples.size > 0:
            pending_samples = np.concatenate([pending_samples, cleaned_samples])

        current_pending_sec = len(pending_samples) / 16000.0

        if i < total_chunks - 1 and current_pending_sec < 5.0:
            print(f"\nChunk {i+1} sau khi lọc còn {current_pending_sec:.2f}s (< 5s), đang gộp với chunk kế tiếp...")
            current_audio_time_ms += chunk_duration_ms
            continue

        with torch.no_grad():
            result = whisper_model.transcribe(
                pending_samples,
                language="vi",
                temperature=[0.0, 0.2, 0.4, 0.6, 0.8],
                beam_size=5,
                best_of=5,
                condition_on_previous_text=True,
                prompt=previous_context if previous_context else None,
                fp16=torch.cuda.is_available(),
                suppress_blank=True
            )

        raw_text = result['text'].strip()
        cleaned_text = detect_and_fix_repetition_realtime(raw_text, previous_context)
        processed_texts.append(cleaned_text)

        words = cleaned_text.split()
        previous_context = " ".join(words[-max_context_words:]) if len(words) > max_context_words else cleaned_text

        end_process_time = time.time()
        processing_duration = end_process_time - start_process_time

        print(f"\nXử lý chunk {i+1}/{total_chunks}")
        print(f"Khung thời gian gốc: {start_sec:.2f}s -> {end_sec:.2f}s")
        print(f"Thời gian thực sau lọc VAD: {current_pending_sec:.2f}s")
        print(f"Thời gian xử lý: {processing_duration:.2f}s")
        print(f"Kết quả: {cleaned_text}")

        pending_samples = np.array([], dtype=np.float32)
        current_audio_time_ms += chunk_duration_ms

    print(f"\nTổng thời lượng sau VAD: {total_speech_duration_sec:.2f} giây")
    return processed_texts

## 7. Gộp các chunk

In [ ]:
def merge_text(processed_texts):
    merged_text = " ".join([text for text in processed_texts if text.strip()]).strip()

    return merged_text

## 8. Sửa văn bản thô

In [ ]:
def detect_and_fix_repetition_realtime(text, previous_context=""):
    if previous_context:
        text = remove_context_overlap(text, previous_context)

    text = remove_immediate_word_repetition(text)
    text = remove_phrase_repetition_advanced(text)

    return text.strip()


def remove_context_overlap(text, previous_context, min_overlap=3):
    text_words = text.split()
    context_words = previous_context.split()

    if len(context_words) < min_overlap or len(text_words) < min_overlap:
        return text

    max_overlap = min(len(context_words), len(text_words), 20)

    for overlap_len in range(max_overlap, min_overlap - 1, -1):
        context_tail = context_words[-overlap_len:]
        text_head = text_words[:overlap_len]

        if context_tail == text_head:
            return " ".join(text_words[overlap_len:])

    return text


def remove_immediate_word_repetition(text, max_consecutive=2):
    words = text.split()
    if not words:
        return text

    result = [words[0]]
    consecutive_count = 1

    for word in words[1:]:
        if word == result[-1]:
            consecutive_count += 1
            if consecutive_count <= max_consecutive:
                result.append(word)
        else:
            result.append(word)
            consecutive_count = 1

    return " ".join(result)


def remove_phrase_repetition_advanced(text, min_phrase_len=2, max_phrase_len=10):
    words = text.split()
    n = len(words)

    if n < min_phrase_len * 2:
        return text

    i = 0
    result = []

    while i < n:
        found_repetition = False

        for phrase_len in range(min(max_phrase_len, n - i), min_phrase_len - 1, -1):
            if i + phrase_len * 2 > n:
                continue

            phrase1 = words[i:i + phrase_len]
            phrase2 = words[i + phrase_len:i + phrase_len * 2]

            if phrase1 == phrase2:
                result.extend(phrase1)
                j = i + phrase_len * 2

                while j + phrase_len <= n and words[j:j + phrase_len] == phrase1:
                    j += phrase_len

                i = j
                found_repetition = True
                break

        if not found_repetition:
            result.append(words[i])
            i += 1

    return " ".join(result)

def remove_fillers(text):
    fillers = {"ờ", "à", "ừ", "ừm", "uh", "ờm", "ơ", "um"}
    words = text.split()
    words = [w for w in words if w.lower() not in fillers]
    return " ".join(words)

def remove_sentence_repetition(text):

    sentences = text.split(". ")
    cleaned = []
    seen = set()

    for s in sentences:

        s_norm = s.strip().lower()

        if s_norm not in seen:
            cleaned.append(s)
            seen.add(s_norm)

    return ". ".join(cleaned)

In [ ]:
def clean_text_with_validation(text):
    cleaned = remove_immediate_word_repetition(text, max_consecutive=2)
    cleaned = remove_phrase_repetition_advanced(cleaned)
    cleaned = remove_sentence_repetition(cleaned)
    cleaned = remove_fillers(cleaned)
    return cleaned

In [ ]:
system_prompt = """
Bạn là một công cụ tự động hiệu đính văn bản (Text Corrector).
Nhiệm vụ DUY NHẤT của bạn là nhận văn bản ASR thô,
sửa lỗi chính tả và điền thêm dấu câu (chấm, phẩy, hỏi chấm)
để câu văn đúng ngữ pháp tiếng Việt.

QUY TẮC TUYỆT ĐỐI (FATAL RULES):
1. GIỮ NGUYÊN 100% ngữ nghĩa và thông tin.
Chỉ sửa lỗi sai chính tả, tuyệt đối không tự ý thêm bớt hay thay đổi từ vựng của người nói.
2. KHÔNG trò chuyện, KHÔNG giải thích, KHÔNG chào hỏi.
3. TUYỆT ĐỐI KHÔNG sử dụng các câu mở đầu như:
"Văn bản sau đây...", "Đây là kết quả...", "Dấu chấm:".
4. ĐẦU RA (OUTPUT) CHỈ ĐƯỢC CHỨA VĂN BẢN ĐÃ SỬA, không chứa bất kỳ ký tự nào khác.
"""
def build_prompt(transcript):
    return f"Input: {transcript}\nOutput:"

In [ ]:
def clean_chunk_with_qwen(text):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_prompt(text)}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        repetition_penalty=1.15,
        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

def split_text_into_chunks(text, max_words=120):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        chunks.append(chunk)
    return chunks

def remove_non_vietnamese(text):
    return re.sub(
        r"[^\u00C0-\u1EF9a-zA-Z0-9\s.,:;!?%]",
        "",
        text
    )

def clean_whisper_transcript(text):
    text = clean_text_with_validation(text)
    chunks = split_text_into_chunks(text)
    cleaned_chunks = []
    for chunk in chunks:
        cleaned = clean_chunk_with_qwen(chunk)
        cleaned = remove_non_vietnamese(cleaned)
        cleaned_chunks.append(cleaned)
    return "\n".join(cleaned_chunks)

## 9. Kết quả

In [ ]:
def transcribe_audio_pipeline(audio_path, whisper_model):
    start_total_time = time.time()

    audio_info = sf.info(audio_path)
    original_duration = audio_info.duration

    print(f"Bắt đầu xử lý: {os.path.basename(audio_path)}")
    print(f"Thời lượng audio gốc: {original_duration:.2f}s")

    print("\n1. Đang xử lý giảm nhiễu...")
    t1 = time.time()
    reduced_audio, sr = reduce_audio_noise(audio_path)
    t2 = time.time()
    print(f"Hoàn thành (Thời gian xử lý: {t2-t1:.2f}s)")

    print("\n2. Đang phân đoạn âm thanh...")
    t3 = time.time()
    chunks = split_audio_into_chunks(reduced_audio, sr, 20)
    t4 = time.time()
    print(f"Có {len(chunks)} đoạn âm thanh. (Thời gian xử lý: {t4-t3:.2f}s)")

    print("\n3. Đang nhận diện âm thanh...")
    t5 = time.time()
    processed_chunks = process_audio_chunks(chunks, whisper_model)
    t6 = time.time()
    print(f"Hoàn thành (Thời gian xử lý: {t6-t5:.2f}s)")

    print("\n4. Đang hoàn thiện văn bản (Qwen LLM)...")
    t7 = time.time()
    raw_text = merge_text(processed_chunks)
    print(f"\nNội dung văn bản sau Whisper:\n{raw_text}")
    final_result = clean_whisper_transcript(raw_text)
    t8 = time.time()
    print(f"Hoàn thành (Thời gian xử lý: {t8-t7:.2f}s)")

    end_total_time = time.time()
    total_duration = end_total_time - start_total_time

    base_name = os.path.splitext(os.path.basename(audio_path))[0]
    output_filename = f"{base_name}.txt"

    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(final_result)

    print(f"\nTệp kết quả: {output_filename}")
    print(f"Thời lượng gốc: {original_duration:.2f}s")
    print(f"Tổng thời gian thực thi: {total_duration:.2f}s")
    print(f"Hiệu suất: {total_duration/original_duration:.2f}x (Thời gian xử lý / Thời lượng audio)")

    return final_result

uploaded = files.upload()

for filename in uploaded.keys():
    result_text = transcribe_audio_pipeline(filename, model_finetuned_small)
    print(f"\nNội dung văn bản cuối cùng:\n{result_text}")

Saving 20 Phút Này Sẽ VĨNH VIỄN THAY ĐỔI Cách Bạn Xem & Hiểu Bóng Đá ! - SOFA Zone - YouTube.mp3 to 20 Phút Này Sẽ VĨNH VIỄN THAY ĐỔI Cách Bạn Xem & Hiểu Bóng Đá ! - SOFA Zone - YouTube.mp3
Bắt đầu xử lý: 20 Phút Này Sẽ VĨNH VIỄN THAY ĐỔI Cách Bạn Xem & Hiểu Bóng Đá ! - SOFA Zone - YouTube.mp3
Thời lượng audio gốc: 1183.71s

1. Đang xử lý giảm nhiễu...
Hoàn thành (Thời gian xử lý: 24.69s)

2. Đang phân đoạn âm thanh...
Có 57 đoạn âm thanh. (Thời gian xử lý: 5.16s)

3. Đang nhận diện âm thanh...

Xử lý chunk 1/57
Khung thời gian gốc: 0.00s -> 20.23s
Thời gian thực sau lọc VAD: 16.50s
Thời gian xử lý: 4.79s
Kết quả: một tiền đạo khi bản những người quyết định bản thẳng lại là người không chẳng mạnh một nội thuộc nữa nhưng sai lợn lớn nhất là đến từ vị trí cách đó ba mươi mét nghe vô lý ư nhưng đó mà cách mống đá bẩn hành tô ních rốt không biết phỏng thượng bút kết chỉ đi bộ mếch cơm là cầu thủ thương mại vân vân và mây mây thì bạn chỉ đang xem mống đá bằng cách

Xử lý chunk 2/57
Khung th